# RSI 분할 매도 + 트레일링 스탑 전략 모델 (Model Version 5)

**기존 모델 대비 개선 사항:**
1. 매수 조건에 RSI, 모멘텀, 거래량 비율, MoM 가속도 추가
2. RSI 단계별 분할 매도 (70/75/80 임계값)
3. 트레일링 스탑 및 손절 로직 추가
4. 동적 백테스팅을 통한 실제 수익률 계산

**Input:**
- symbol: 종목 코드
- surprise_z: ARIMA 수출 서프라이즈 Z-Score
- gics_code: GICS Sector 코드

**Output:**
- decision: 'BUY', 'SELL', 'HOLD'
- Expected_Returns: 보유 기간별 예상 평균 수익률
  - Avg_Return_Post_1D (%)
  - Avg_Return_Post_2D (%)
  - Avg_Return_Post_5D (%)
  - Avg_Return_Post_10D (%)
  - Avg_Return_Post_20D (%)
  - RSI_Trailing_Return (%): RSI 분할매도 + 트레일링스탑 전략 수익률 (신규 추가)

## 1. 라이브러리 임포트 및 경로 설정

In [47]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# 경로 설정
DATA_DIR = "../../data"
OUTPUT_DIR = "../../output/model_rsi_trailing"
MODEL_FILE_NAME = "model_rsi_trailing.pkl"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 데이터 파일 경로
EXPORT_SURPRISE_PATH = f"../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
CLOSE_PRICE_PATH = f"{DATA_DIR}/price/close.csv"
VOLUME_PATH = f"{DATA_DIR}/price/volume.csv"
GICS_SECTOR_PATH = f"{DATA_DIR}/gics/sector.csv"

print("✅ 경로 설정 완료")

✅ 경로 설정 완료


## 2. 모델 하이퍼파라미터 정의

In [48]:
# 매수 조건
BUY_CONDITIONS = {
    'zscore_yoy_min': 2.0,          # YoY Z-Score 최소값 (기존 모델과 동일)
    'mom_min': 5.0,                  # MoM 최소값 (%)
    'positive_acceleration': True,   # MoM 가속도 > 0 조건
    'momentum_20_min': 8.0,          # 20일 모멘텀 최소값 (%)
    'volume_ratio_min': 2.0,         # 거래량 비율 최소값 (평균 대비)
    'rsi_range': (50, 75),           # RSI 범위 (과매수 회피)
    'sectors': [15, 20, 45]          # 타겟 섹터 (소재, 산업재, 커뮤니케이션)
}

# RSI 분할 매도 임계값
RSI_THRESHOLDS = {
    'rsi_70': 0.2,  # RSI ≥ 70: 20% 매도
    'rsi_75': 0.3,  # RSI ≥ 75: 30% 추가 매도
    'rsi_80': 1.0   # RSI ≥ 80: 나머지 전량 매도
}

# 손절 및 트레일링 스탑
STOP_LOSS_PCT = -10.0           # 손절: -10%
TRAILING_STOP_PCT = -10.0       # 트레일링 스탑: 최고점 대비 -10%
MAX_HOLDING_DAYS = 20           # 최대 보유 기간

print("✅ 하이퍼파라미터 설정 완료")
print(f"   매수 조건: {BUY_CONDITIONS}")
print(f"   RSI 임계값: {RSI_THRESHOLDS}")
print(f"   손절: {STOP_LOSS_PCT}%, 트레일링: {TRAILING_STOP_PCT}%, 최대 보유: {MAX_HOLDING_DAYS}일")

✅ 하이퍼파라미터 설정 완료
   매수 조건: {'zscore_yoy_min': 2.0, 'mom_min': 5.0, 'positive_acceleration': True, 'momentum_20_min': 8.0, 'volume_ratio_min': 2.0, 'rsi_range': (50, 75), 'sectors': [15, 20, 45]}
   RSI 임계값: {'rsi_70': 0.2, 'rsi_75': 0.3, 'rsi_80': 1.0}
   손절: -10.0%, 트레일링: -10.0%, 최대 보유: 20일


## 3. 기술적 지표 계산 함수 정의

In [49]:
def calculate_rsi(prices: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    """RSI (Relative Strength Index) 계산"""
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period, min_periods=7).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period, min_periods=7).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calculate_momentum(prices: pd.DataFrame, period: int = 20) -> pd.DataFrame:
    """모멘텀 계산: N일 전 대비 변화율 (%)"""
    return prices.pct_change(period, fill_method=None) * 100


def calculate_volume_ratio(volume: pd.DataFrame, window: int = 20) -> pd.DataFrame:
    """거래량 비율 계산: 현재 거래량 / N일 이동평균"""
    volume_ma = volume.rolling(window=window, min_periods=10).mean()
    return volume / volume_ma

print("✅ 기술적 지표 계산 함수 정의 완료")

✅ 기술적 지표 계산 함수 정의 완료


## 4. 데이터 로드 및 전처리

In [50]:
print("="*100)
print("RSI 분할 매도 + 트레일링 스탑 전략 모델 (Model Version 5)")
print("="*100)

print("\n[1/7] 데이터 로딩...")

try:
    # 수출 Surprise 데이터
    surprise_df = pd.read_csv(EXPORT_SURPRISE_PATH)
    surprise_df['date'] = pd.to_datetime(surprise_df['date'])

    # 주가 데이터
    close_df = pd.read_csv(CLOSE_PRICE_PATH, index_col=0)
    close_df.index = pd.to_datetime(close_df.index.astype(str), format='%Y%m%d')

    volume_df = pd.read_csv(VOLUME_PATH, index_col=0)
    volume_df.index = pd.to_datetime(volume_df.index.astype(str), format='%Y%m%d')

    # GICS 섹터 매핑
    sector_df = pd.read_csv(GICS_SECTOR_PATH)
    sector_map = dict(zip(sector_df['symbol'], sector_df['value']))
    surprise_df['sector'] = surprise_df['symbol'].map(sector_map)

    print(f"   ✅ 수출 Surprise 데이터: {len(surprise_df)} rows")
    print(f"   ✅ 주가 데이터: {close_df.shape}")
    print(f"   ✅ 거래량 데이터: {volume_df.shape}")
    
    # 데이터 미리보기
    print("\n[데이터 미리보기]")
    display(surprise_df.head())

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다: {e}")
    raise

RSI 분할 매도 + 트레일링 스탑 전략 모델 (Model Version 5)

[1/7] 데이터 로딩...
   ✅ 수출 Surprise 데이터: 10443 rows
   ✅ 주가 데이터: (1231, 349)
   ✅ 거래량 데이터: (1231, 349)

[데이터 미리보기]


,date,symbol,surprise_z,close_price,return_post_1d,return_post_2d,return_post_5d,return_post_10d,return_post_20d,sector
0,2020-06-30,AEGQRD,-0.054030,4245.103607,-0.007092,-0.011820,0.007092,-0.016548,-0.016548,35.0
1,2020-07-31,AEGQRD,0.133750,3984.015231,-0.011990,-0.007194,-0.045564,-0.069544,0.270983,35.0
2,2020-08-31,AEGQRD,0.205735,5125.068136,0.062366,0.169892,0.440860,0.210753,0.070968,35.0
3,2020-11-30,AEGQRD,-0.004405,4883.319639,0.042770,0.020367,0.044807,0.004073,-0.040733,35.0
4,2021-03-31,AEGQRD,-0.058449,4863.979760,0.017510,0.019455,0.103113,0.108949,0.108949,35.0


## 5. MoM 가속도 계산

In [51]:
surprise_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10443 entries, 0 to 10442
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date             10443 non-null  datetime64[ns]
 1   symbol           10443 non-null  object        
 2   surprise_z       10443 non-null  float64       
 3   close_price      10443 non-null  float64       
 4   return_post_1d   10440 non-null  float64       
 5   return_post_2d   10440 non-null  float64       
 6   return_post_5d   10440 non-null  float64       
 7   return_post_10d  10440 non-null  float64       
 8   return_post_20d  10440 non-null  float64       
 9   sector           10233 non-null  float64       
dtypes: datetime64[ns](1), float64(8), object(1)
memory usage: 816.0+ KB


In [52]:
import pandas as pd
import numpy as np
import os
from typing import final

# --- 경로 설정 (필요시 수정) ---
# NOTE: 원본 수출액 데이터가 저장된 경로를 지정해야 합니다.
EXPORT_VALUE_PATH = "../../output/export_value_clean.csv" 


def calculate_mom_metrics(surprise_df: pd.DataFrame) -> pd.DataFrame:
    """
    원본 수출액 데이터를 병합하고, MoM, MoM_change, MoM_acceleration을 계산합니다.
    """
    
    # 1. 원본 수출액 데이터 로드
    try:
        df_export = pd.read_csv(EXPORT_VALUE_PATH, usecols=['symbol', 'date', 'export_value'])
        df_export['date'] = pd.to_datetime(df_export['date'])
    except FileNotFoundError:
        print(f"❌ 오류: 원본 수출액 파일({EXPORT_VALUE_PATH})을 찾을 수 없습니다. MoM 계산을 건너뜁니다.")
        return surprise_df.assign(mom=np.nan, mom_change=np.nan, mom_acceleration=np.nan)

    # 2. 'export_value'를 현재 데이터프레임에 병합
    df_merged = pd.merge(
        surprise_df,
        df_export,
        on=['symbol', 'date'],
        how='left' # 기존 surprise_df의 모든 행을 유지
    )
    
    if 'export_value' not in df_merged.columns:
        print("❌ 오류: 'export_value' 병합 실패. 날짜 및 종목 코드를 확인하세요.")
        return surprise_df.assign(mom=np.nan, mom_change=np.nan, mom_acceleration=np.nan)

    # 3. MoM (전월 대비 변화율) 계산
    df_merged = df_merged.sort_values(['symbol', 'date']).reset_index(drop=True)
    df_merged['mom'] = df_merged.groupby('symbol')['export_value'].pct_change()
    
    # 4. MoM Change (MoM의 1차 차분)
    df_merged['mom_change'] = df_merged.groupby('symbol')['mom'].diff()
    
    # 5. MoM Acceleration (MoM Change의 1차 차분)
    df_merged['mom_acceleration'] = df_merged.groupby('symbol')['mom_change'].diff()

    # 6. 임시로 추가한 'export_value' 컬럼 제거 후 반환
    return df_merged.drop(columns=['export_value'], errors='ignore')


# --- 실행: surprise_df에 MoM 가속도 계산 적용 ---
# NOTE: 이 코드를 실행하기 전에 'surprise_df' 변수가 정의되어 있어야 합니다.
# (이전 코드에서 df_analysis 대신 surprise_df를 사용하셨으므로 변수명을 유지합니다.)

print("\n[2/7] MoM 가속도 계산...")
# assume surprise_df is the current DataFrame
surprise_df = calculate_mom_metrics(surprise_df) 

print("   ✅ MoM 가속도 계산 완료")
print("\n[MoM 가속도 통계]")
# 계산된 새로운 컬럼으로 describe 실행
print(surprise_df[['symbol', 'date', 'mom', 'mom_change', 'mom_acceleration']].describe())


[2/7] MoM 가속도 계산...
   ✅ MoM 가속도 계산 완료

[MoM 가속도 통계]
                                date            mom     mom_change  \
count                          10443   10100.000000    9757.000000   
mean   2022-11-25 07:36:58.328066560      58.645644      -0.118800   
min              2020-06-30 00:00:00      -0.999709 -537542.170250   
25%              2021-11-30 00:00:00      -0.193149      -0.375918   
50%              2023-01-31 00:00:00       0.011437      -0.019831   
75%              2023-11-30 00:00:00       0.259804       0.368723   
max              2024-10-31 00:00:00  537542.441176  537543.439637   
std                              NaN    5357.605001    7709.299409   

       mom_acceleration  
count      9.416000e+03  
mean      -5.748582e+01  
min       -1.075086e+06  
25%       -6.105865e-01  
50%        7.060634e-02  
75%        7.736761e-01  
max        5.375427e+05  
std        1.241220e+04  


In [53]:
import pandas as pd
import numpy as np
import os
from typing import final

# --- 경로 설정 (원본 수출액 파일 경로) ---
# NOTE: 이 경로는 원본 'export_value.csv' 파일의 위치를 가정합니다.
EXPORT_VALUE_PATH = "../../output/export_value_clean.csv" 


def calculate_mom_metrics(surprise_df: pd.DataFrame) -> pd.DataFrame:
    """
    원본 수출액 데이터를 병합하고, MoM, MoM_change, MoM_acceleration을 계산합니다.
    """
    
    # 1. 원본 수출액 데이터 로드
    try:
        df_export = pd.read_csv(EXPORT_VALUE_PATH, usecols=['symbol', 'date', 'export_value'])
        df_export['date'] = pd.to_datetime(df_export['date'])
    except FileNotFoundError:
        print(f"❌ 오류: 원본 수출액 파일({EXPORT_VALUE_PATH})을 찾을 수 없습니다. MoM 계산을 건너뜁니다.")
        return surprise_df.assign(mom=np.nan, mom_change=np.nan, mom_acceleration=np.nan)

    # 2. 'export_value'를 현재 데이터프레임에 병합
    # NOTE: df_analysis/surprise_df에는 이미 date가 datetime 형식이라고 가정합니다.
    df_merged = pd.merge(
        surprise_df,
        df_export,
        on=['symbol', 'date'],
        how='left' # 기존 surprise_df의 모든 행을 유지
    )
    
    if 'export_value' not in df_merged.columns:
        print("❌ 오류: 'export_value' 병합 실패. 날짜 및 종목 코드를 확인하세요.")
        return surprise_df.assign(mom=np.nan, mom_change=np.nan, mom_acceleration=np.nan)

    # 3. MoM (전월 대비 변화율) 계산
    df_merged = df_merged.sort_values(['symbol', 'date']).reset_index(drop=True)
    df_merged['mom'] = df_merged.groupby('symbol')['export_value'].pct_change()
    
    # 4. MoM Change (MoM의 1차 차분)
    df_merged['mom_change'] = df_merged.groupby('symbol')['mom'].diff()
    
    # 5. MoM Acceleration (MoM Change의 1차 차분)
    df_merged['mom_acceleration'] = df_merged.groupby('symbol')['mom_change'].diff()

    # 6. 임시로 추가한 'export_value' 컬럼 제거 후 반환
    return df_merged.drop(columns=['export_value'], errors='ignore')


# --- 실행: surprise_df에 MoM 가속도 계산 적용 ---

# NOTE: 이 코드는 'surprise_df' 변수가 정의되어 있어야 실행 가능합니다.
# (이전 과정의 로드 코드가 실행되었다고 가정합니다.)

print("\n[2/7] MoM 가속도 계산...")
# surprise_df 변수가 정의되어 있어야 합니다. (이전 셀에서 로드되어야 함)
surprise_df = calculate_mom_metrics(surprise_df) 

print("   ✅ MoM 가속도 계산 완료")
print("\n[MoM 가속도 통계]")
print(surprise_df[['symbol', 'date', 'mom', 'mom_change', 'mom_acceleration']].describe())


[2/7] MoM 가속도 계산...
   ✅ MoM 가속도 계산 완료

[MoM 가속도 통계]
                                date            mom     mom_change  \
count                          10443   10100.000000    9757.000000   
mean   2022-11-25 07:36:58.328066560      58.645644      -0.118800   
min              2020-06-30 00:00:00      -0.999709 -537542.170250   
25%              2021-11-30 00:00:00      -0.193149      -0.375918   
50%              2023-01-31 00:00:00       0.011437      -0.019831   
75%              2023-11-30 00:00:00       0.259804       0.368723   
max              2024-10-31 00:00:00  537542.441176  537543.439637   
std                              NaN    5357.605001    7709.299409   

       mom_acceleration  
count      9.416000e+03  
mean      -5.748582e+01  
min       -1.075086e+06  
25%       -6.105865e-01  
50%        7.060634e-02  
75%        7.736761e-01  
max        5.375427e+05  
std        1.241220e+04  


In [54]:
import pandas as pd

# 1️⃣ export_value.csv 불러오기
export_df = pd.read_csv("../../output/export_value_clean.csv")[["symbol", "export_value"]]

# 2️⃣ 중복된 symbol이 있다면 평균 등으로 하나로 합치기 (선택사항)
# export_df = export_df.groupby("symbol", as_index=False)["export_value"].mean()

# 3️⃣ symbol 기준으로 병합 (export_value만 추가됨)
surprise_df = surprise_df.merge(export_df, on="symbol", how="left")

# 4️⃣ 확인
print(surprise_df.head())




        date  symbol  surprise_z  close_price  return_post_1d  return_post_2d  \
0 2020-06-30  AEGQRD    -0.05403  4245.103607       -0.007092        -0.01182   
1 2020-06-30  AEGQRD    -0.05403  4245.103607       -0.007092        -0.01182   
2 2020-06-30  AEGQRD    -0.05403  4245.103607       -0.007092        -0.01182   
3 2020-06-30  AEGQRD    -0.05403  4245.103607       -0.007092        -0.01182   
4 2020-06-30  AEGQRD    -0.05403  4245.103607       -0.007092        -0.01182   

   return_post_5d  return_post_10d  return_post_20d  sector  mom  mom_change  \
0        0.007092        -0.016548        -0.016548    35.0  NaN         NaN   
1        0.007092        -0.016548        -0.016548    35.0  NaN         NaN   
2        0.007092        -0.016548        -0.016548    35.0  NaN         NaN   
3        0.007092        -0.016548        -0.016548    35.0  NaN         NaN   
4        0.007092        -0.016548        -0.016548    35.0  NaN         NaN   

   mom_acceleration  export_valu

In [55]:
# 'export_value' 컬럼이 존재한다고 가정하고 MoM (전월 대비 변화율)을 계산합니다.
# 이 계산을 통해 'mom' 컬럼이 생성되어 KeyError가 해결됩니다.
print("\n[2/7] MoM 가속도 계산...")

# 1. 'mom' (MoM Change) 계산: 'export_value'를 기준으로 백분율 변화(pct_change) 계산
# NOTE: export_value가 시계열 데이터의 주축이라고 가정합니다.
surprise_df['mom'] = surprise_df.groupby('symbol')['export_value'].pct_change()

# 2. 'mom_change' 계산: MoM 변화율 (MoM의 1차 차분)
surprise_df['mom_change'] = surprise_df.groupby('symbol')['mom'].diff()

# 3. 'mom_acceleration' 계산: MoM 변화율의 변화 (MoM의 2차 차분)
surprise_df['mom_acceleration'] = surprise_df.groupby('symbol')['mom_change'].diff()

print("   ✅ MoM 가속도 계산 완료")
print("\n[MoM 가속도 통계]")
print(surprise_df[['symbol', 'date', 'mom', 'mom_change', 'mom_acceleration']].describe())


[2/7] MoM 가속도 계산...
   ✅ MoM 가속도 계산 완료

[MoM 가속도 통계]
                                date            mom     mom_change  \
count                         701886  701543.000000  701200.000000   
mean   2022-11-15 21:45:04.900795904      44.588050      -0.000471   
min              2020-06-30 00:00:00      -0.999998 -991521.120028   
25%              2021-11-30 00:00:00      -0.194420      -0.371870   
50%              2022-11-30 00:00:00       0.006597      -0.012791   
75%              2023-11-30 00:00:00       0.252889       0.358616   
max              2024-10-31 00:00:00  991521.250000  991522.249799   
std                              NaN    6054.450731    8564.615326   

       mom_acceleration  
count      7.008570e+05  
mean       4.028370e-04  
min       -1.983043e+06  
25%       -6.069352e-01  
50%        5.227752e-02  
75%        7.554831e-01  
max        9.915223e+05  
std        1.483798e+04  


In [56]:
print("\n[2/7] MoM 가속도 계산...")

surprise_df = surprise_df.sort_values(['symbol', 'date']).reset_index(drop=True)
surprise_df['mom_change'] = surprise_df.groupby('symbol')['mom'].diff()
surprise_df['mom_acceleration'] = surprise_df.groupby('symbol')['mom_change'].diff()

print("   ✅ MoM 가속도 계산 완료")
print("\n[MoM 가속도 통계]")
print(surprise_df[['symbol', 'date', 'mom', 'mom_change', 'mom_acceleration']].describe())


[2/7] MoM 가속도 계산...
   ✅ MoM 가속도 계산 완료

[MoM 가속도 통계]
                                date            mom     mom_change  \
count                         701886  701543.000000  701200.000000   
mean   2022-11-15 21:45:04.900795904      44.588050      -0.000471   
min              2020-06-30 00:00:00      -0.999998 -991521.120028   
25%              2021-11-30 00:00:00      -0.194420      -0.371870   
50%              2022-11-30 00:00:00       0.006597      -0.012791   
75%              2023-11-30 00:00:00       0.252889       0.358616   
max              2024-10-31 00:00:00  991521.250000  991522.249799   
std                              NaN    6054.450731    8564.615326   

       mom_acceleration  
count      7.008570e+05  
mean       4.028370e-04  
min       -1.983043e+06  
25%       -6.069352e-01  
50%        5.227752e-02  
75%        7.554831e-01  
max        9.915223e+05  
std        1.483798e+04  


## 6. 기술적 지표 계산

In [57]:
print("\n[3/7] 기술적 지표 계산...")

momentum_20 = calculate_momentum(close_df, period=20)
volume_ratio = calculate_volume_ratio(volume_df, window=20)
rsi = calculate_rsi(close_df, period=14)

print("   ✅ RSI, 모멘텀, 거래량 비율 계산 완료")
print("\n[RSI 통계]")
print(rsi.describe())


[3/7] 기술적 지표 계산...
   ✅ RSI, 모멘텀, 거래량 비율 계산 완료

[RSI 통계]
            AEGQRD       AIJFBS       AJAAJF       ANKPII      AOWNEI  \
count  1225.000000  1225.000000  1225.000000  1225.000000  466.000000   
mean     47.653212    48.110692    48.829187    49.283522   42.240194   
std      16.273816    17.119752    16.518471    15.841807   16.598312   
min       3.750000     6.493506     5.769231     6.143345    0.000000   
25%      36.542670    36.363636    36.170213    38.161560   30.769231   
50%      48.888889    47.619048    48.936170    49.679487   44.452713   
75%      59.304348    59.409378    60.839161    60.427807   54.228607   
max      96.494845    94.339623    91.139241    89.396171   82.256020   

            AOWXSX       APJJHM       ASANPU       ASEXYC       ATUPVQ  ...  \
count  1225.000000  1225.000000  1225.000000  1225.000000  1225.000000  ...   
mean     46.801293    50.847640    51.166652    50.872408    50.559284  ...   
std      18.058861    17.396215    15.992799   

## 7. 매수 시그널 탐지

In [58]:
print("\n[4/7] 매수 시그널 탐지...")

buy_signals = []

for idx, row in surprise_df.iterrows():
    try:
        ann_date = row['date']
        symbol = row['symbol']
        zscore_yoy = row['rolling_zscore_yoy']
        mom = row['mom']
        mom_accel = row['mom_acceleration']
        sector = row['sector']

        # 조건 1: Z-Score 임계값
        if pd.isna(zscore_yoy) or zscore_yoy < BUY_CONDITIONS['zscore_yoy_min']:
            continue

        # 조건 2: 섹터 필터
        if pd.isna(sector) or sector not in BUY_CONDITIONS['sectors']:
            continue

        # 조건 3: MoM 최소값
        if pd.isna(mom) or mom < BUY_CONDITIONS['mom_min']:
            continue

        # 조건 4: MoM 가속도 양수
        if pd.isna(mom_accel) or mom_accel <= 0:
            continue

        # 매수 거래일 찾기 (공시일 다음날부터 10일 이내)
        next_month = ann_date + pd.DateOffset(days=1)
        trade_dates = close_df.index[
            (close_df.index >= next_month) &
            (close_df.index <= next_month + pd.Timedelta(days=10))
        ]

        if len(trade_dates) == 0 or symbol not in close_df.columns:
            continue

        trade_date = trade_dates[0]

        # 조건 5: 20일 모멘텀
        mom20 = momentum_20.loc[trade_date, symbol]
        if pd.isna(mom20) or mom20 < BUY_CONDITIONS['momentum_20_min']:
            continue

        # 조건 6: 거래량 비율
        vol_ratio = volume_ratio.loc[trade_date, symbol]
        if pd.isna(vol_ratio) or vol_ratio < BUY_CONDITIONS['volume_ratio_min']:
            continue

        # 조건 7: RSI 범위
        rsi_val = rsi.loc[trade_date, symbol]
        if pd.isna(rsi_val):
            continue
        rsi_min, rsi_max = BUY_CONDITIONS['rsi_range']
        if rsi_val < rsi_min or rsi_val > rsi_max:
            continue

        # 진입가
        entry_price = close_df.loc[trade_date, symbol]
        if pd.isna(entry_price) or entry_price == 0:
            continue

        buy_signals.append({
            'ann_date': ann_date,
            'trade_date': trade_date,
            'symbol': symbol,
            'sector': sector,
            'entry_price': entry_price,
            'zscore_yoy': zscore_yoy,
            'mom': mom,
            'mom_acceleration': mom_accel,
            'entry_rsi': rsi_val
        })

    except Exception as e:
        continue

buy_signals_df = pd.DataFrame(buy_signals)
print(f"   ✅ 매수 시그널: {len(buy_signals_df)}개 발견")

if len(buy_signals_df) > 0:
    print("\n[매수 시그널 미리보기]")
    display(buy_signals_df.head(10))
else:
    print("\n❌ 매수 시그널이 없어 백테스팅을 수행할 수 없습니다.")
    print("   조건을 완화하거나 데이터 기간을 확인하세요.")


[4/7] 매수 시그널 탐지...
   ✅ 매수 시그널: 0개 발견

❌ 매수 시그널이 없어 백테스팅을 수행할 수 없습니다.
   조건을 완화하거나 데이터 기간을 확인하세요.


## 8. RSI 분할 매도 + 트레일링 스탑 백테스팅

In [ ]:
print("\n[5/7] RSI 분할 매도 + 트레일링 스탑 백테스팅...")

results = []

for idx, buy in buy_signals_df.iterrows():
    symbol = buy['symbol']
    trade_date = buy['trade_date']
    entry_price = buy['entry_price']

    future_dates = close_df.index[close_df.index > trade_date][:MAX_HOLDING_DAYS]

    # 상태 추적
    position = 1.0  # 보유 비중 (1.0 = 100%)
    max_price = entry_price  # 최고점
    total_return = 0.0  # 누적 수익률
    sell_events = []  # 매도 이벤트 기록

    # RSI 단계별 매도 플래그
    sold_at_70 = False
    sold_at_75 = False
    sold_at_80 = False

    for day_idx, future_date in enumerate(future_dates):
        holding_days = day_idx + 1

        try:
            current_price = close_df.loc[future_date, symbol]
            if pd.isna(current_price) or current_price == 0:
                continue

            # 최고점 갱신
            if current_price > max_price:
                max_price = current_price

            current_return = (current_price / entry_price - 1) * 100
            current_rsi = rsi.loc[future_date, symbol]

            # 1. 손절: -10% 도달 시 전량 매도
            if current_return <= STOP_LOSS_PCT and position > 0:
                sell_return = current_return * position
                total_return += sell_return
                sell_events.append({
                    'day': holding_days,
                    'reason': '손절 (-10%)',
                    'sell_ratio': position,
                    'price': current_price,
                    'return': sell_return
                })
                position = 0
                break

            # 2. 트레일링 스탑: 최고점 대비 -10% 하락 시 전량 매도
            drawdown_from_peak = (current_price / max_price - 1) * 100
            if drawdown_from_peak <= TRAILING_STOP_PCT and position > 0:
                sell_return = current_return * position
                total_return += sell_return
                sell_events.append({
                    'day': holding_days,
                    'reason': f'트레일링 스탑 (최고점 대비 {drawdown_from_peak:.1f}%)',
                    'sell_ratio': position,
                    'price': current_price,
                    'return': sell_return
                })
                position = 0
                break

            # 3. RSI 단계별 분할 매도
            if pd.notna(current_rsi):
                # RSI ≥ 80: 나머지 전량 매도
                if current_rsi >= 80 and not sold_at_80 and position > 0:
                    sell_ratio = position
                    sell_return = current_return * sell_ratio
                    total_return += sell_return
                    sell_events.append({
                        'day': holding_days,
                        'reason': f'RSI 80 이상 ({current_rsi:.1f})',
                        'sell_ratio': sell_ratio,
                        'price': current_price,
                        'return': sell_return
                    })
                    position = 0
                    sold_at_80 = True
                    break

                # RSI ≥ 75: 30% 추가 매도
                elif current_rsi >= 75 and not sold_at_75 and position > 0:
                    sell_ratio = 0.3
                    sell_return = current_return * sell_ratio
                    total_return += sell_return
                    position -= sell_ratio
                    sell_events.append({
                        'day': holding_days,
                        'reason': f'RSI 75 이상 ({current_rsi:.1f})',
                        'sell_ratio': sell_ratio,
                        'price': current_price,
                        'return': sell_return
                    })
                    sold_at_75 = True

                # RSI ≥ 70: 20% 매도
                elif current_rsi >= 70 and not sold_at_70 and position > 0:
                    sell_ratio = 0.2
                    sell_return = current_return * sell_ratio
                    total_return += sell_return
                    position -= sell_ratio
                    sell_events.append({
                        'day': holding_days,
                        'reason': f'RSI 70 이상 ({current_rsi:.1f})',
                        'sell_ratio': sell_ratio,
                        'price': current_price,
                        'return': sell_return
                    })
                    sold_at_70 = True

        except:
            continue

    # 30일 경과 또는 마지막 날에 남은 포지션 청산
    if position > 0 and len(future_dates) > 0:
        last_date = future_dates[-1]
        last_price = close_df.loc[last_date, symbol]
        if not pd.isna(last_price):
            last_return = (last_price / entry_price - 1) * 100
            sell_return = last_return * position
            total_return += sell_return
            sell_events.append({
                'day': len(future_dates),
                'reason': f'{MAX_HOLDING_DAYS}일 경과',
                'sell_ratio': position,
                'price': last_price,
                'return': sell_return
            })
            position = 0

    # 고정 보유 수익률 계산 (기존 모델과 비교용)
    fixed_returns = {}
    for days in [1, 2, 5, 10, 20]:
        if days <= len(future_dates):
            exit_price = close_df.loc[future_dates[days-1], symbol]
            if not pd.isna(exit_price):
                fixed_returns[f'return_post_{days}d'] = (exit_price / entry_price - 1) * 100

    # 결과 저장
    if len(sell_events) > 0:
        results.append({
            'symbol': symbol,
            'trade_date': trade_date,
            'sector': buy['sector'],
            'entry_price': entry_price,
            'rsi_trailing_return': total_return,
            'num_sells': len(sell_events),
            **fixed_returns
        })

results_df = pd.DataFrame(results)
print(f"   ✅ 백테스트 완료: {len(results_df)}개 거래")

if len(results_df) > 0:
    print("\n[백테스트 결과 미리보기]")
    display(results_df.head(10))

## 9. 모델 성능 검증

In [ ]:
print("\n[6/7] 모델 성능 검증...")

if len(results_df) > 0:
    # 전략별 평균 수익률 계산
    strategy_comparison = {
        'RSI 분할매도 + 트레일링': results_df['rsi_trailing_return'].mean(),
    }

    for days in [1, 2, 5, 10, 20]:
        col = f'return_post_{days}d'
        if col in results_df.columns:
            valid_data = results_df[results_df[col].notna()][col]
            if len(valid_data) > 0:
                strategy_comparison[f'고정 {days}일 보유'] = valid_data.mean()

    # DataFrame으로 변환
    comparison_df = pd.DataFrame({
        'Strategy': list(strategy_comparison.keys()),
        'Avg_Return (%)': list(strategy_comparison.values())
    })

    print("\n" + "="*85)
    print("전략별 평균 수익률 비교")
    print("="*85)
    display(comparison_df)
    print("="*85)

    # Expected Returns 형식으로 출력
    expected_returns = {
        'Avg_Return_Post_1D (%)': results_df['return_post_1d'].mean() if 'return_post_1d' in results_df.columns else 0,
        'Avg_Return_Post_2D (%)': results_df['return_post_2d'].mean() if 'return_post_2d' in results_df.columns else 0,
        'Avg_Return_Post_5D (%)': results_df['return_post_5d'].mean() if 'return_post_5d' in results_df.columns else 0,
        'Avg_Return_Post_10D (%)': results_df['return_post_10d'].mean() if 'return_post_10d' in results_df.columns else 0,
        'Avg_Return_Post_20D (%)': results_df['return_post_20d'].mean() if 'return_post_20d' in results_df.columns else 0,
        'RSI_Trailing_Return (%)': results_df['rsi_trailing_return'].mean()
    }

    print("\n" + "="*85)
    print("예상 수익률 (Expected Returns) - BUY 시그널 평균")
    print("="*85)
    expected_returns_df = pd.DataFrame({
        'Metric': list(expected_returns.keys()),
        'Value (%)': [f"{v:.4f}" for v in expected_returns.values()]
    })
    display(expected_returns_df)
    print("="*85)
else:
    expected_returns = {}

## 10. 모델 규칙 저장

In [ ]:
print("\n[7/7] 모델 규칙 저장...")

model_rules = {
    'model_name': 'RSI_Gradual_Trailing_Stop_Classifier',
    'version': 'v5.0',
    'buy_conditions': BUY_CONDITIONS,
    'rsi_thresholds': RSI_THRESHOLDS,
    'stop_loss_pct': STOP_LOSS_PCT,
    'trailing_stop_pct': TRAILING_STOP_PCT,
    'max_holding_days': MAX_HOLDING_DAYS,
    'expected_returns': expected_returns if len(results_df) > 0 else {}
}

model_path = os.path.join(OUTPUT_DIR, MODEL_FILE_NAME)
with open(model_path, 'wb') as f:
    pickle.dump(model_rules, f)

print(f"\n✅ 모델 규칙 저장 완료: {model_path}")

# 결과 CSV 저장
if len(results_df) > 0:
    results_csv_path = os.path.join(OUTPUT_DIR, "backtest_results.csv")
    results_df.to_csv(results_csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ 백테스트 결과 저장 완료: {results_csv_path}")

print("\n" + "="*100)
print("모델 Version 5 실행 완료!")
print("="*100)

## 11. 최종 요약 및 투자 결정 함수

In [ ]:
def get_investment_decision_rsi_trailing(symbol: str, surprise_z: float, gics_code: int) -> Dict[str, Any]:
    """
    RSI 분할매도 + 트레일링스탑 전략 기반 투자 결정
    
    Input:
        - symbol: 종목 코드
        - surprise_z: ARIMA 수출 서프라이즈 Z-Score
        - gics_code: GICS Sector 코드
    
    Output:
        - decision: 'BUY', 'SELL', 'HOLD'
        - Expected_Returns: 보유 기간별 예상 수익률
    """
    # 간단한 의사결정 로직 (실제로는 더 복잡한 조건 필요)
    if surprise_z > BUY_CONDITIONS['zscore_yoy_min'] and gics_code in BUY_CONDITIONS['sectors']:
        decision = 'BUY'
    elif surprise_z < -BUY_CONDITIONS['zscore_yoy_min']:
        decision = 'SELL'
    else:
        decision = 'HOLD'
    
    return {
        'decision': decision,
        'Expected_Returns': expected_returns if len(results_df) > 0 else {}
    }

# 테스트
print("\n[투자 결정 함수 테스트]")
test_result = get_investment_decision_rsi_trailing(
    symbol='A005930',
    surprise_z=2.5,
    gics_code=15
)
print(f"Decision: {test_result['decision']}")
print(f"Expected Returns: {test_result['Expected_Returns']}")